# Setup

## Import

In [2]:
import pathlib
from typing import Tuple, Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split, KFold, cross_val_score
# from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression

# Custom
from utils.base import load_dataset
from utils.eda import check_dataset_size_and_shape, check_columns_for_nulls

## Constants

In [3]:
DATASET_FILENAME = "Barclays_Tweets_BarclaysTweets.csv"

## Utils

## Load Dataset

In [4]:

df = load_dataset(DATASET_FILENAME)
df_shape: Tuple[int, int] = df.shape

# Exploratory Data Analysis (EDA)

In [5]:
check_dataset_size_and_shape(df)

Number of rows: 364
Dataset shape: (364, 7)
Number of rows (alternative): 364


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 364 entries, 0 to 363
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RowID            364 non-null    int64  
 1   message_id       364 non-null    str    
 2   date             364 non-null    str    
 3   tweet_text       364 non-null    str    
 4   positive         0 non-null      float64
 5   negative         0 non-null      float64
 6   Sentiment_score  0 non-null      float64
dtypes: float64(3), int64(1), str(3)
memory usage: 20.0 KB


In [7]:
df.describe()

,RowID,positive,negative,Sentiment_score
count,364.000000,0.0,0.0,0.0
mean,182.500000,NaN,NaN,NaN
std,105.221988,NaN,NaN,NaN
min,1.000000,NaN,NaN,NaN
25%,91.750000,NaN,NaN,NaN
50%,182.500000,NaN,NaN,NaN
75%,273.250000,NaN,NaN,NaN
max,364.000000,NaN,NaN,NaN


In [8]:
df.head()

,RowID,message_id,date,tweet_text,positive,negative,Sentiment_score
0,1,2.13103E+17,6/13/2012,I love the barclays feedback machines that say...,NaN,NaN,NaN
1,2,2.13335E+17,6/14/2012,Get even more Jay Z There will be a 40 40 Club...,NaN,NaN,NaN
2,3,2.13686E+17,6/15/2012,RbS sharesup 7 9 Lloyds up 5 2 Barclays up 4 2...,NaN,NaN,NaN
3,4,2.13664E+17,6/15/2012,Jay Z To Open 40 40 Club at brooklyn s Barclay...,NaN,NaN,NaN
4,5,2.14184E+17,6/16/2012,my life would be complete if i went to a world...,NaN,NaN,NaN


## Missing Data

Expected missing from last 3 attributes as these will be populated.

In [9]:
check_columns_for_nulls(df)

RowID              False
message_id         False
date               False
tweet_text         False
positive            True
negative            True
Sentiment_score     True
dtype: bool


# Sentiment Analysis

In [10]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

nltk.download('vader_lexicon', quiet=True)

sia = SentimentIntensityAnalyzer()

def get_sentiment_scores(text: str) -> Dict[str, float]:
    scores = sia.polarity_scores(str(text))
    return scores['pos'], scores['neg'], scores['compound'], scores['neu']

df[['positive', 'negative', 'Sentiment_score', 'neutral']] = df['tweet_text'].apply(
    lambda x: pd.Series(get_sentiment_scores(x))
)

df[['tweet_text', 'positive', 'negative', 'Sentiment_score', 'neutral']].head(10)

,tweet_text,positive,negative,Sentiment_score,neutral
0,I love the barclays feedback machines that say...,0.280,0.000,0.7650,0.720
1,Get even more Jay Z There will be a 40 40 Club...,0.000,0.000,0.0000,1.000
2,RbS sharesup 7 9 Lloyds up 5 2 Barclays up 4 2...,0.256,0.000,0.7351,0.744
3,Jay Z To Open 40 40 Club at brooklyn s Barclay...,0.000,0.000,0.0000,1.000
4,my life would be complete if i went to a world...,0.000,0.000,0.0000,1.000
5,send the brotha much love from the Tube,0.375,0.000,0.6369,0.625
6,Much Luv Fam,0.000,0.000,0.0000,1.000
7,"excellent vid, definitely poor management. the...",0.388,0.188,0.5106,0.424
8,thanks,1.000,0.000,0.4404,0.000
9,yup. I remember the trouble he gave Trinidad f...,0.000,0.252,-0.4019,0.748


# Text Analysis

## Text Classification

In [11]:
def classify_sentiment(score: float) -> str:
    if score >= 0.05:
        return 'Positive'
    elif score <= -0.05:
        return 'Negative'
    return 'Neutral'

df['text_class'] = df['Sentiment_score'].apply(classify_sentiment)
print(df['text_class'].value_counts())
df[['tweet_text', 'Sentiment_score', 'text_class']].head(10)

text_class
Positive    138
Negative    132
Neutral      94
Name: count, dtype: int64


,tweet_text,Sentiment_score,text_class
0,I love the barclays feedback machines that say...,0.7650,Positive
1,Get even more Jay Z There will be a 40 40 Club...,0.0000,Neutral
2,RbS sharesup 7 9 Lloyds up 5 2 Barclays up 4 2...,0.7351,Positive
3,Jay Z To Open 40 40 Club at brooklyn s Barclay...,0.0000,Neutral
4,my life would be complete if i went to a world...,0.0000,Neutral
5,send the brotha much love from the Tube,0.6369,Positive
6,Much Luv Fam,0.0000,Neutral
7,"excellent vid, definitely poor management. the...",0.5106,Positive
8,thanks,0.4404,Positive
9,yup. I remember the trouble he gave Trinidad f...,-0.4019,Negative


## Topic Extraction

Uses LDA (Latent Dirichlet Allocation).

In [12]:
# Check if scikit-learn is installed and can be imported
# Ran into issues but after a kernel restart it seems to be working fine. This is just a sanity check.
import sklearn
print("Scikit-learn version:", sklearn.__version__)

try:
    from sklearn.feature_extraction.text import CountVectorizer
    print("Success! CountVectorizer imported properly.")
except ImportError as e:
    print("Import failed with error:", e)


Scikit-learn version: 1.9.0
Success! CountVectorizer imported properly.


In [15]:
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)  # remove URLs
    text = re.sub(r'@\w+', '', text)             # remove @mentions
    text = re.sub(r'[^a-z\s]', '', text)         # letters only
    return text.strip()

df['clean_text'] = df['tweet_text'].apply(clean_text)

n_topics = 5
count_vec = CountVectorizer(stop_words='english', max_features=500, min_df=2)
dtm = count_vec.fit_transform(df['clean_text'])

lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
lda.fit(dtm)

feature_names = count_vec.get_feature_names_out()
for idx, topic in enumerate(lda.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-9:-1]]
    print(f"Topic {idx+1}: {', '.join(top_words)}")

df['topic'] = lda.transform(dtm).argmax(axis=1)
df[['tweet_text', 'topic']].head(10)

Topic 1: barclays, diamond, libor, bob, scandal, pay, man, don
Topic 2: barclays, bob, diamond, just, finance, bankers, great, london
Topic 3: fine, people, work, like, criminals, away, corrupt, way
Topic 4: barclays, diamond, uk, boss, quits, right, bob, fraud
Topic 5: barclays, libor, banking, scandal, bank, chairman, banks, diamond


,tweet_text,topic
0,I love the barclays feedback machines that say...,4
1,Get even more Jay Z There will be a 40 40 Club...,4
2,RbS sharesup 7 9 Lloyds up 5 2 Barclays up 4 2...,1
3,Jay Z To Open 40 40 Club at brooklyn s Barclay...,4
4,my life would be complete if i went to a world...,1
5,send the brotha much love from the Tube,4
6,Much Luv Fam,3
7,"excellent vid, definitely poor management. the...",1
8,thanks,0
9,yup. I remember the trouble he gave Trinidad f...,3


## Text Clustering

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

n_clusters = 5
tfidf = TfidfVectorizer(stop_words='english', max_features=300, min_df=2)
tfidf_matrix = tfidf.fit_transform(df['clean_text'])  # reuses clean_text from above

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(tfidf_matrix)

print(df['cluster'].value_counts().sort_index())

terms = tfidf.get_feature_names_out()
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]
for i in range(n_clusters):
    top = [terms[ind] for ind in order_centroids[i, :8]]
    print(f"Cluster {i}: {', '.join(top)}")

df[['tweet_text', 'cluster']].head(10)

cluster
0    123
1     52
2     41
3    138
4     10
Name: count, dtype: int64
Cluster 0: barclays, libor, bank, banks, rates, new, says, premier
Cluster 1: diamond, bob, barclays, ceo, blog, says, boss, quits
Cluster 2: scandal, chairman, barclays, agius, marcus, banking, quits, libor
Cluster 3: just, love, like, great, time, lol, people, man
Cluster 4: center, jay, brooklyn, barclays, concerts, play, open, pm


,tweet_text,cluster
0,I love the barclays feedback machines that say...,3
1,Get even more Jay Z There will be a 40 40 Club...,4
2,RbS sharesup 7 9 Lloyds up 5 2 Barclays up 4 2...,0
3,Jay Z To Open 40 40 Club at brooklyn s Barclay...,4
4,my life would be complete if i went to a world...,0
5,send the brotha much love from the Tube,3
6,Much Luv Fam,3
7,"excellent vid, definitely poor management. the...",3
8,thanks,3
9,yup. I remember the trouble he gave Trinidad f...,3


### Eval

In [20]:
from collections import defaultdict
from time import time

from sklearn import metrics

evaluations = []
evaluations_std = []

labels = df['clean_text']


def fit_and_evaluate(km, X, name=None, n_runs=5):
    name = km.__class__.__name__ if name is None else name

    train_times = []
    scores = defaultdict(list)
    for seed in range(n_runs):
        km.set_params(random_state=seed)
        t0 = time()
        km.fit(X)
        train_times.append(time() - t0)
        scores["Homogeneity"].append(metrics.homogeneity_score(labels, km.labels_))
        scores["Completeness"].append(metrics.completeness_score(labels, km.labels_))
        scores["V-measure"].append(metrics.v_measure_score(labels, km.labels_))
        scores["Adjusted Rand-Index"].append(
            metrics.adjusted_rand_score(labels, km.labels_)
        )
        scores["Silhouette Coefficient"].append(
            metrics.silhouette_score(X, km.labels_, sample_size=2000)
        )
    train_times = np.asarray(train_times)

    print(f"clustering done in {train_times.mean():.2f} ± {train_times.std():.2f} s ")
    evaluation = {
        "estimator": name,
        "train_time": train_times.mean(),
    }
    evaluation_std = {
        "estimator": name,
        "train_time": train_times.std(),
    }
    for score_name, score_values in scores.items():
        mean_score, std_score = np.mean(score_values), np.std(score_values)
        print(f"{score_name}: {mean_score:.3f} ± {std_score:.3f}")
        evaluation[score_name] = mean_score
        evaluation_std[score_name] = std_score
    evaluations.append(evaluation)
    evaluations_std.append(evaluation_std)

In [21]:
fit_and_evaluate(kmeans, tfidf_matrix, name="KMeans\non tf-idf vectors")

clustering done in 0.04 ± 0.00 s 
Homogeneity: 0.187 ± 0.037
Completeness: 1.000 ± 0.000
V-measure: 0.313 ± 0.053
Adjusted Rand-Index: 0.000 ± 0.000
Silhouette Coefficient: 0.038 ± 0.005


## Deep Categorization

In [17]:
categories = [
    'account issue',
    'payment or transaction problem',
    'customer service complaint',
    'fraud or security concern',
    'product or service inquiry',
    'general feedback',
]

try:
    from transformers import pipeline
    classifier = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')
    texts = df['tweet_text'].fillna('').tolist()
    # Limit to first 50 — model is slow; remove slice for full dataset
    results = [classifier(t, categories)['labels'][0] for t in texts[:50]]
    df.loc[df.index[:50], 'deep_category'] = results
    print('Zero-shot classification done (first 50 rows)')

except ImportError:
    print('transformers not installed — using keyword fallback')
    keyword_map = {
        'account issue':                  ['account', 'login', 'password', 'access', 'locked'],
        'payment or transaction problem':  ['payment', 'transfer', 'transaction', 'charge', 'fee'],
        'customer service complaint':      ['help', 'support', 'service', 'wait', 'agent', 'poor'],
        'fraud or security concern':       ['fraud', 'scam', 'unauthorized', 'stolen', 'hack'],
        'product or service inquiry':      ['app', 'card', 'bank', 'product', 'offer', 'rate'],
    }
    def keyword_categorize(text):
        t = str(text).lower()
        for cat, kws in keyword_map.items():
            if any(kw in t for kw in kws):
                return cat
        return 'general feedback'
    df['deep_category'] = df['tweet_text'].apply(keyword_categorize)

print(df['deep_category'].value_counts())
df[['tweet_text', 'deep_category']].head(10)

transformers not installed — using keyword fallback
deep_category
general feedback                  234
product or service inquiry         87
customer service complaint         13
fraud or security concern          13
payment or transaction problem     10
account issue                       7
Name: count, dtype: int64


,tweet_text,deep_category
0,I love the barclays feedback machines that say...,payment or transaction problem
1,Get even more Jay Z There will be a 40 40 Club...,general feedback
2,RbS sharesup 7 9 Lloyds up 5 2 Barclays up 4 2...,product or service inquiry
3,Jay Z To Open 40 40 Club at brooklyn s Barclay...,general feedback
4,my life would be complete if i went to a world...,general feedback
5,send the brotha much love from the Tube,general feedback
6,Much Luv Fam,general feedback
7,"excellent vid, definitely poor management. the...",customer service complaint
8,thanks,general feedback
9,yup. I remember the trouble he gave Trinidad f...,general feedback
